## Asignación de nuevas observaciones mediante los Intervalos de Confianza (IC)

- Autor: Germán Homero Morán Figueroa
- Descripción: En este notebook se realiza la implementación de funciones y metodos que permiten realizar la asignación de nuevos 
                registros (conjunto de validación) mediante las predicciones de los intervalos de confianza



In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import mean_absolute_error,max_error,mean_squared_error
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy import stats
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.regression.linear_model as linear_model
from statsmodels.tools.tools import pinv_extended  
from scipy import stats
import joblib


In [3]:
# Funciones
# ==================================================================

'''
Esta Función permite el calculo de la regresión lineal para cada agrupación. En este caso se tienen
3 salidas:
- model: Modelo de regresión 
- r2: coeficiente de dterminación
- yhat: y_predicho


'''
class LinearRegession():
    def __init__(self, df, alpha, l1_ratio):
        self.df = df
        self.alpha = alpha
        self.l1_ratio = l1_ratio
    

    def CalcularModeloLR(self):
        # alpha=0.1, l1_ratio=0.97
        Y = self.df.RDT_AJUSTADO.values
        X = self.df.drop(["RDT_AJUSTADO","ID_LOTE"], axis=1).values 
        modelElasticNet = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio, random_state=123)
        model = modelElasticNet.fit(X,Y)
        #r_2 = model.score(X,Y)
        # Pedicciones
        yhat = model.predict(X)
        r_2 = r2_score(Y, yhat)
    
        
        return [model, r_2, yhat]



'''
Esta clase es una nueva versión del modelo de regresión Lineal. En este caso se tienen las siguientes salidas:
- r2: coeficeinte de determinación
- coef: coeficientes del modelo
- intercept: intercepto del modelo
- 
'''
class LinearRegessionv2():
    def __init__(self, df, alpha, l1_ratio):
        self.df = df
        self.alpha = alpha
        self.l1_ratio = l1_ratio
    

    def CalcularModeloLR(self):
        # alpha=0.1, l1_ratio=0.97
        Y = self.df.RDT_AJUSTADO.values
        X = self.df.drop(["RDT_AJUSTADO","ID_LOTE"], axis=1).values 
        modelElasticNet = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio, random_state=123)
        model = modelElasticNet.fit(X,Y)
        #r_2 = model.score(X,Y)
        # Pedicciones
        yhat = model.predict(X)
        r_2 = r2_score(Y, yhat)
        coef = model.coef_
        intercep = model.intercept_
        residuals = np.power((Y-yhat),2)
        ssr = sum(residuals)

        return  r_2, coef, intercep,ssr,residuals
    

# Funciones apra el calculo de los Intervalos de Confianza
"""
model: Modelo entrenado
x: Valor a testear (X_test)
varianza residual: 
"""
def predict_with_confidence_intervals(model, X, residual_variance, confidence=0.95):
    # Predicciones
    preds = model.predict(X)

    # Grados de libertad
    n = X.shape[0]
    print("n...",n)
    p = X.shape[1]
    print("p....",p)
    df = n - p - 1

    
    # Error estándar de las predicciones
    se_pred = np.sqrt(residual_variance * (1 + np.sum((X - np.mean(X, axis=0))**2 / np.var(X, axis=0), axis=1)))
    print("Error Estandar", se_pred)
    # Valor crítico para el intervalo de confianza
    t_value = stats.t.ppf((1 + confidence) / 2., df)
    print("t_value: ", t_value)
    # Intervalos de confianza
    ci_upper = preds + t_value * se_pred
    ci_lower = preds - t_value * se_pred

    return preds, ci_lower, ci_upper



'''
Función para cargar los grupos de los modelos previamente entrenados

'''
def cargarGrupos():
    # Cargamos los respectivos grupos
    listaGruposDefinitivos = []
    G0 = pd.read_excel("DataOpt/grupo_N0.xlsx")
    G1 = pd.read_excel("DataOpt/grupo_N1.xlsx")
    G2 = pd.read_excel("DataOpt/grupo_N2.xlsx")

    print(f"Longitud G0: ",{len(G0)},"longitud G1: ", {len(G1)} , "Longitud G2: ", {len(G2)})

    # Eliminamos clumnas
    G0 = G0.drop(["Unnamed: 0"], axis=1)
    G1 = G1.drop(["Unnamed: 0"], axis=1)
    G2 = G2.drop(["Unnamed: 0"], axis=1)
    listaGruposDefinitivos.append(G0)
    listaGruposDefinitivos.append(G1)
    listaGruposDefinitivos.append(G2)

    return listaGruposDefinitivos



'''
Función para cargar la correlación asociada a c/u de los grupos definidos
'''
def CalcularCorrelationInitial(gruposDefinitivos):
    correlacionesIniciales=[]
    for i in range(len(gruposDefinitivos)):
        r_2 = LinearRegessionv2(gruposDefinitivos[i], 0.1, 0.97).CalcularModeloLR()
        correlacionesIniciales.append(r_2)
    
    return correlacionesIniciales



'''
Función para realizar las predicciones con el conjunto de testeo, segun los modelo obtenidos.
Entradas:
        - test_clean: df sin rdt_ajustado y ID_LOTE
        - lista_modelos : En formato .pkl
'''
def PredctionsModels(lista_modelos, test_clean):
    predictions = []
    for i in range(len(lista_modelos)):
        psi_pred = []
        print("Prediciones Modelo ", i)
        for j in range(len(test_clean)):
            pred = lista_modelos[i].predict(test_clean.values[j].reshape(1,-1))
            psi_pred.append(pred[0])
        
        predictions.append(psi_pred)
    return predictions



''' 
Funcion para realizar el cargue de los modelos en formato .pkl  

'''
def CargueModelos():
    lista_modelos = []
    for i in range(3):
        model = joblib.load(f'DataOpt/IC/modelo_ols_{i}.pkl') 
        lista_modelos.append(model)
    print(len(lista_modelos))
    return lista_modelos


def agregarPredicitions(test_original, predictions):
    df0 = test_original.copy()
    df1= test_original.copy()
    df2 = test_original.copy()
    df0.RDT_AJUSTADO = predictions[0]
    df1.RDT_AJUSTADO = predictions[1]
    df2.RDT_AJUSTADO = predictions[2]
    list_dfs = list([df0,df1,df2])
    return list_dfs

In [4]:
'''
    Función que permite calcular los intervalos de confianza de cada predicción
    y determinar si los los limites se ajustan a los modelos definidos.
'''

def IntervalosConfianza(df_entrenamiento, df_testeo):
    Y_train = df_entrenamiento.RDT_AJUSTADO.values
    X_train= df_entrenamiento.drop(["RDT_AJUSTADO","ID_LOTE"], axis=1).values
    Y_test =df_testeo.RDT_AJUSTADO.values
    X_test = df_testeo.drop(["RDT_AJUSTADO","ID_LOTE"], axis=1).values
    # Agreago Constantes al modelo
    X_train = sm.add_constant(X_train, prepend=True)
    X_test = sm.add_constant(X_test, prepend=True)
    model = sm.OLS(Y_train, X_train)
    res= model.fit_regularized(method='elastic_net',alpha=0.1, L1_wt=0.97)
    model_fit_regularized = model.fit(params=res.params)
    predicciones = model_fit_regularized.predict(exog = X_test)
    # Calculo de los intervalos de confianza
    IC = model_fit_regularized.get_prediction(exog = X_test, )
    Sumary_IC = IC.summary_frame(alpha=0.1)
    Sumary_IC["Predicciones"] = predicciones 
    df_summary = Sumary_IC[["mean_ci_lower","mean_ci_upper","Predicciones"]]

    return df_summary



'''
Función para calcular los intervalos de confianza de cada grupo (N)
'''
def calculoIntervalosConfianza(grupos_definitivos, test_original):
    lista_df_IC = []
    for i in range(len(grupos_definitivos)):
        IC = IntervalosConfianza(grupos_definitivos[i],test_original)
        lista_df_IC.append(IC)
    return lista_df_IC




'''
Función para calcular los dataframes finales asociados a sus respectivos Intervalos de confianza
'''

lista_aux_var_ic = ["mean_ci_lower","mean_ci_upper"]
def DfWithConfidenceIntervals(listaIntervalosConfianza, test_original,lista_aux_var_ic):
    lista_df_ic = []
    for i in range(len(listaIntervalosConfianza)):
        print("Grupo: ", i)
        for j in range(len(lista_aux_var_ic)):
            G_IC= test_original.copy()
            G_IC["RDT_AJUSTADO"] = listaIntervalosConfianza[i][lista_aux_var_ic[j]]
            lista_df_ic.append([G_IC,i])
    return lista_df_ic


def GruposFinales(li):
    lista_modificada = []
    for i in range(len(li)):
        if li[i] == 0:
            lista_modificada.append(0)
        if li[i] == 1:
            lista_modificada.append(0)
        if li[i] == 2:
            lista_modificada.append(1)
        if li[i] == 3:
            lista_modificada.append(1)
        if li[i] == 4:
            lista_modificada.append(2)
        if li[i] == 5:
            lista_modificada.append(2)
    return lista_modificada



def LinearRegessionOLMS(df_entrenamiento):
    Y = df_entrenamiento.RDT_AJUSTADO.values
    X = df_entrenamiento.drop(["RDT_AJUSTADO","ID_LOTE"], axis=1).values     
    # Agreago Constantes al modelo
    X_train = sm.add_constant(X, prepend=True)
    model = sm.OLS(Y, X_train)
    res= model.fit_regularized(method='elastic_net',alpha=0.1, L1_wt=0.97)
    model_fit_regularized = model.fit(params=res.params)
    r_2 = model_fit_regularized.rsquared
    print("Nuevo r_2: ", r_2)
    return r_2, model_fit_regularized


def CalcularCorrelationInitialOLS(gruposDefinitivos):
    correlacionesIniciales=[]
    for i in range(len(gruposDefinitivos)):        
        r_2, model = LinearRegessionOLMS(gruposDefinitivos[i])
        # Guardo Correlacion 
        correlacionesIniciales.append(r_2)
        # Gurado modelos
        name_model = f'DataOpt/IC/modelo_ols_{i}.pkl'
        joblib.dump(model,name_model) # Guardo el modelo.  
    return correlacionesIniciales





'''
Metricas de desempeño del modelo CLR
'''

def metricasPerformanceCLR(y_true, yhat_test):
    R2 = r2_score(y_true, yhat_test)
    MAE =  mean_absolute_error(y_true, yhat_test)
    ME = max_error(y_true, yhat_test)
    MSE = mean_squared_error(y_true, yhat_test)
    print("Coeficiente de determinación : ", R2)
    print("Error absoluto medio: ", MAE)
    print("Maximo error residual: ", ME)
    print("RMSE: ", MSE)


In [39]:
'''
G0_Lower: DF 
list_corr: Lista de las correaciones originales para c/grupos
'''

def calculosTotalCorelaciones(G0_lower, lista_corr_olsms, indice_lista_corr ,grupos_definitivos):
    # Obtengo las correlaciones de c/d registro
    var_corr = []
    for i in range(len(G0_lower)):
        NG_LW = grupos_definitivos[indice_lista_corr].append(G0_lower.iloc[i],ignore_index = True)
        # Obtengo el R2 nuevo
        #new_r2= LinearRegessionv2(NG_LW,0.1, 0.97).CalcularModeloLR()
        new_r2,model= LinearRegessionOLMS(NG_LW)
        # Obtengo la diferencia de variaciones
        diff_corr = lista_corr_olsms[indice_lista_corr]- new_r2
        var_corr.append(diff_corr)
        # Eliminamos el registro del grupo (ultima posicion)
        NG_LW.drop([len(NG_LW) -1],axis=0, inplace=True)
        #print("Longitud df_final: ", NG_LW.shape)
    
    return var_corr
           


def FinalCorreationsDf(lista_df_ic, list_corr, grupos_definitivos):
    lista_corr_grups = []
    for i in range(len(lista_df_ic)):
        df_lw_up = lista_df_ic[i][0]
        indice_lista_corr = lista_df_ic[i][1]
        print("Indice: ", indice_lista_corr)
        c_df = calculosTotalCorelaciones(df_lw_up, list_corr, indice_lista_corr,grupos_definitivos)
        lista_corr_grups.append(c_df)
    
    return lista_corr_grups



'''
Entradas: lista de todas als correlaciones asociadas a cada df:
salidad: Grupo final donde el resgistro causa menos impacto.

'''
def groupsAsociateCorrelation(fff):
    df_final_correations = pd.DataFrame(fff)
    ## O incluso eliminarlo
    #pd.options.display.max_columns = None
    aa = df_final_correations.min()
    # Encuentro el indice asociado correlacion minima
    li = list(df_final_correations.idxmin())
    lista_modificada = GruposFinales(li)

    return lista_modificada


'''
Desempeño del modelo teniendo en cuenta la asignacion final de los modelos
En esta función se calcula el y_predicho de acuerdo al grupo asignado
'''
def performanceModel(listAsigFinales,lista_modelos, test_clean):
    y_pred= []
    for j in range(len(test_clean)):
        number_model = listAsigFinales[j]
        pred = lista_modelos[number_model].predict(test_clean.values[j].reshape(1,-1))
        y_pred.append(pred[0])
    
    return y_pred




In [37]:
# Lectura del conjunto de validacion 
# ============================================================================
test = pd.read_csv("DataOpt/dataset_test_Original.csv")
test.head(5)
test_original = test.copy()
test_clean = test.copy()
test_clean = test.drop(["ID_LOTE","RDT_AJUSTADO"], axis=1)
print("Dimenciones conjunto de test: ", test_clean.shape)
test_clean.head(5)

Dimenciones conjunto de test:  (40, 174)


,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,ContMalQui_Antes_Siem,ContMalQui_Siem_Emer,ContMalQui_Emer_Flor,ContMalQui_Flor_Cose,ContPlaQui_Antes_Siem,ContPlaQui_Siem_Emer,ContPlaQui_Emer_Flor,ContPlaQui_Flor_Cose,TotN_Antes_Siem,TotN_Siem_Emer,TotN_Emer_Flor,TotP_Antes_Siem,TotP_Siem_Emer,TotP_Emer_Flor,TotK_Antes_Siem,TotK_Siem_Emer,TotK_Emer_Flor,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor,PENDIENTE_RASTA,NO_CAPAS_RASTA,PH_RASTA,PROFUND_CAP_ENDURE_RASTA,ESPESOR_CAP_ENDURE_RASTA,PROFUND_MOTEADOS_RASTA,PROFUND_RAICES_VIVAS_RASTA,prof_efectiva,Porc_A,Porc_Ar,Porc_ArA,Porc_ArL,Porc_FrL,Porc_L,Porc_F,porc.x,porc.y,Porc_AF,Porc_BLANDO,Porc_DURO,Porc_EXT_DURO,Porc_FRIABLE,Porc_FIRME,Porc_EXT_FIRME,Porc_PLASTICO,Porc_MUY_PLASTICO,Temp_Max_Avg_Veg,Temp_Min_Avg_Veg,Temp_Avg_Veg,Diurnal_Range_Avg_Veg,Sol_Ener_Accu_Veg,Temp_Max_34_Freq_Veg,Rain_Accu_Veg,Rain_10_Freq_Veg,Rhum_Avg_Veg,Temp_Max_Avg_For,Temp_Min_Avg_For,Temp_Avg_For,Diurnal_Range_Avg_For,Sol_Ener_Accu_For,Temp_Max_34_Freq_For,Rain_Accu_For,Rain_10_Freq_For,Rhum_Avg_For,Temp_Max_Avg_Mad,Temp_Min_Avg_Mad,Temp_Avg_Mad,Diurnal_Range_Avg_Mad,Sol_Ener_Accu_Mad,Temp_Max_34_Freq_Mad,Rain_Accu_Mad,Rain_10_Freq_Mad,Rhum_Avg_Mad,TIPO_SIEMBRA_Manual,TIPO_SIEMBRA_Mecanizado,MATERIAL_GENETICO_ADV 9293 (Syngenta),MATERIAL_GENETICO_ADV 9339 (Syngenta),MATERIAL_GENETICO_CORPOICA V 114,MATERIAL_GENETICO_Cerato (Syngenta),MATERIAL_GENETICO_DK 1040,MATERIAL_GENETICO_DK 1596,MATERIAL_GENETICO_DK 234,MATERIAL_GENETICO_DK 234 YGRR,MATERIAL_GENETICO_DK7088,MATERIAL_GENETICO_FNC 114,MATERIAL_GENETICO_FNC 3056,MATERIAL_GENETICO_ICA V 109,MATERIAL_GENETICO_ICA V 156,MATERIAL_GENETICO_ICA V 305,MATERIAL_GENETICO_Impacto (Syngenta),MATERIAL_GENETICO_NK254 (Syngenta),MATERIAL_GENETICO_Otro,MATERIAL_GENETICO_P3966 (Pioneer),MATERIAL_GENETICO_P4082 (Pioneer),MATERIAL_GENETICO_PAC 105,MATERIAL_GENETICO_PIONEER 30F32,MATERIAL_GENETICO_PIONEER 30F32HW,MATERIAL_GENETICO_PIONEER 30F35,MATERIAL_GENETICO_PIONEER 30F35 H,MATERIAL_GENETICO_PIONEER 30F35 HRR,MATERIAL_GENETICO_Sinko (Syngenta),MATERIAL_GENETICO_Status (Syngenta),CULT_ANT_Algodon,CULT_ANT_Frijol,CULT_ANT_Maiz,CULT_ANT_Pastos,CULT_ANT_Yuca,METODO_COSECHA_Manual,METODO_COSECHA_Mecanizada,TERRENO_CIRCUN_RASTA_ONDULADO,TERRENO_CIRCUN_RASTA_ONDULADO Y MONTANIOSO,TERRENO_CIRCUN_RASTA_PLANO O LLANO,POSICION_PERFIL_RASTA_LADERA CONCAVA,POSICION_PERFIL_RASTA_LADERA CONVEXA,POSICION_PERFIL_RASTA_LADERA PLANA,POSICION_PERFIL_RASTA_PIE DE UNA ELEVACION,POSICION_PERFIL_RASTA_PLANO,POSICION_PERFIL_RASTA_PLANO CON ONDULACIONES,PEDREG_PERFIL_ROCAS_MUY ROCOSO,PEDREG_PERFIL_ROCAS_SIN ROCAS,ESTRUCTURA_RASTA_ATERRONADA,ESTRUCTURA_RASTA_GRANULAR,ESTRUCTURA_RASTA_MASIVA,ESTRUCTURA_RASTA_SUELTA O POLVOSA,OBSERVA_COSTRAS_DURAS_RASTA_MUY MARCADAS,OBSERVA_COSTRAS_DURAS_RASTA_NO HAY,OBSERVA_COSTRAS_DURAS_RASTA_POCO MARCADAS,SITIO_EXPUESTO_SOL_RASTA_LA MANIANA,SITIO_EXPUESTO_SOL_RASTA_LA MANIANA Y LA TARDE,SITIO_EXPUESTO_SOL_RASTA_LA TARDE,OBSERVA_COSTRAS_BLANCAS_RASTA_NO HAY,OBSERVA_COSTRAS_BLANCAS_RASTA_POCO MARCADAS,OBSERVA_COSTAS_NEGRAS_RASTA_NO HAY,OBSERVA_COSTAS_NEGRAS_RASTA_POCO MARCADAS,REGION_SECA_ARIDA_RASTA_NO,REGION_SECA_ARIDA_RASTA_SI,OBSERVA_PLANTAS_PEQUENAS_RASTA_MUY AFECTADAS,OBSERVA_PLANTAS_PEQUENAS_RASTA_NO HAY CULTIVO,OBSERVA_PLANTAS_PEQUENAS_RASTA_PLANTAS NORMALES,OBSERVA_PLANTAS_PEQUENAS_RASTA_POCO AFECTADAS,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_BUENO,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_ESPACIADO,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_MUY BUENO,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_REGULAR,RECUBRIMIENTO_VEGETAL__SUELO_RASTA_SIN COBERTURA,d.interno_BUENO,d.interno_EXCESIVO,d.interno_LENTO A MUY LENTO,drenaje_externo_LENTO,drenaje_externo_NINGUNO,SEM_TRATADAS,DRENAJE,ALMACENAMIENTO_FINCA,CAP_ENDURE_RASTA,MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA

In [11]:
# 2. Cargue Grupos Definitivos durante el entrenamiento 
# ========================================================================
grupos_definitivos = cargarGrupos()

# Calculo de correlaciones finales de los grupos
# =========================================================================
lista_corr_olsms = CalcularCorrelationInitialOLS(grupos_definitivos)
print("lista Correlaciones Iniciales: ", lista_corr_olsms)
 
 # Cargue modelos finales del entrenamiento
 # ======================================================================
list_modelos = CargueModelos()

# Calculos de los intervalos para las predicciones segun cada modelo
# =======================================================================
listaIntervalosConfianza = calculoIntervalosConfianza(grupos_definitivos, test_original)

# Calculo de los datafames  definitivos con sus intervalos de confianza 
# ========================================================================
lista_aux_var_ic = ["mean_ci_lower","mean_ci_upper"]
lista_df_ic = DfWithConfidenceIntervals(listaIntervalosConfianza, test_original,lista_aux_var_ic)

Longitud G0:  {264} longitud G1:  {260} Longitud G2:  {235}
Nuevo r_2:  0.9535896398727237
Nuevo r_2:  0.9506674486861093
Nuevo r_2:  0.9456447959278168
lista Correlaciones Iniciales:  [0.9535896398727237, 0.9506674486861093, 0.9456447959278168]
3
Grupo:  0
Grupo:  1
Grupo:  2


In [22]:
# Calculo de las correlaciones finales para cada grupo
# =========================================================================================
list_final_correlations = FinalCorreationsDf(lista_df_ic, lista_corr_olsms,grupos_definitivos)

Indice:  0


<ipython-input-21-9cab10ddb7a7>:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  NG_LW = grupos_definitivos[indice_lista_corr].append(G0_lower.iloc[i],ignore_index = True)


Nuevo r_2:  0.9533841335747891


<ipython-input-21-9cab10ddb7a7>:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  NG_LW = grupos_definitivos[indice_lista_corr].append(G0_lower.iloc[i],ignore_index = True)


Nuevo r_2:  0.9544866102930641


...

Nuevo r_2:  0.9450137697237208


<ipython-input-21-9cab10ddb7a7>:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  NG_LW = grupos_definitivos[indice_lista_corr].append(G0_lower.iloc[i],ignore_index = True)


Nuevo r_2:  0.9454079799049966


<ipython-input-21-9cab10ddb7a7>:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  NG_LW = grupos_definitivos[indice_lista_corr].append(G0_lower.iloc[i],ignore_index = True)


Nuevo r_2:  0.9455191924435593


In [40]:
# Calculo del menor impacto en las correlaciones segun el intervalo de confianza de cada predicción 
# ====================================================================================================
correlacionesFinales = groupsAsociateCorrelation(list_final_correlations)

# Calculo del rendimeinto predicho segun el grupo donde causa menos impacto
# ====================================================================================================
y_pred = performanceModel(correlacionesFinales,list_modelos, test_clean)
metricasPerformanceCLR(test_original.RDT_AJUSTADO[0:10], y_pred[0:10])

Coeficiente de determinación :  -0.271797123677076
Error absoluto medio:  1156.1772182500686
Maximo error residual:  2679.2110490260284
RMSE:  1976741.4629447064
